In [ ]:
4. LSTM for Part of Speech tagging: Implement LSTM-based model for Parts of Speech (POS) tagging.
Given a sequence of words, train the LSTM network to predict the corresponding POS tags by learning
word dependencies and context from a labeled dataset.

What you did:
Trained deep learning model for POS tagging
🎯 Why we do it:

To learn context automatically using neural networks.

🌍 Use in NLP:
Advanced NLP systems
Chatbots
Language modeling

In [3]:
# Step 1: Import
import pandas as pd# Step 1: Import

# Import pandas for data handling (tables, datasets)
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Tokenizer converts text into numerical sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# pad_sequences ensures all sequences have equal length
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sequential model to build neural network layer-by-layer
from tensorflow.keras.models import Sequential

# Import layers:
# Embedding → converts words to vectors
# LSTM → captures sequence patterns
# TimeDistributed → applies layer to each time step
# Dense → fully connected output layer
from tensorflow.keras.layers import Embedding, LSTM, TimeDistributed, Dense

# Split dataset into training and testing sets
from sklearn.model_selection import train_test_split
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, TimeDistributed, Dense
from sklearn.model_selection import train_test_split

In [4]:
# Step 2: Load & Clean Data
df = pd.read_csv("ner_datasetreference.csv", encoding="latin1")
df.head()

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O


In [5]:
# Fill missing values in the "Sentence #" column
# Forward fill (ffill) copies the previous value downward
df["Sentence #"] = df["Sentence #"].ffill()

# Fill missing values in the "Word" column
# Backward fill (bfill) copies the next value upward
df["Word"] = df["Word"].bfill()

In [6]:
# Step 3: Create Sentences

# Group words by each sentence and combine them into a list
# Result: [["I", "love", "coding"], ...]
sentences = df.groupby("Sentence #")["Word"].apply(list).values

# Group corresponding POS tags for each sentence
# Result: [["PRON", "VERB", "NOUN"], ...]
tags = df.groupby("Sentence #")["POS"].apply(list).values

In [7]:
# Step 4: Tokenization

# Create tokenizer for input words
tok = Tokenizer()

# Learn vocabulary from sentences
tok.fit_on_texts(sentences)

# Convert words into numerical sequences
X = tok.texts_to_sequences(sentences)

# Create tokenizer for POS tags (output labels)
tag_tok = Tokenizer()

# Learn tag vocabulary
tag_tok.fit_on_texts(tags)

# Convert tags into numerical sequences
y = tag_tok.texts_to_sequences(tags)

In [8]:
# Step 5: Padding

# Define maximum length for all sequences
MAX_LEN = 128

# Pad input sequences with zeros at the end to make equal length
X = pad_sequences(X, maxlen=MAX_LEN, padding='post')

# Pad output tag sequences similarly
y = pad_sequences(y, maxlen=MAX_LEN, padding='post')

In [9]:
# Step 6: Train-Test Split

# Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [10]:
# Step 7: Model

# Create a sequential deep learning model
model = Sequential([
    
    # Embedding layer: converts word indices into dense vector representations
    Embedding(len(tok.word_index)+1, 64),
    
    # LSTM layer: learns sequential patterns in the data
    # return_sequences=True → output for each time step (needed for tagging)
    LSTM(64, return_sequences=True),
    
    # TimeDistributed Dense layer:
    # Applies Dense layer to each time step for predicting POS tags
    TimeDistributed(Dense(np.max(y)+1, activation='softmax'))
])

# Compile the model:
# optimizer='adam' → efficient optimization algorithm
# loss='sparse_categorical_crossentropy' → used for multi-class classification with integer labels
# metrics=['accuracy'] → track accuracy during training
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [11]:
# Step 8: Train

# Train the model on training data
# epochs=5 → number of times the model sees the entire dataset
# batch_size=32 → number of samples processed at once
model.fit(X_train, y_train, epochs=5, batch_size=32)

Epoch 1/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 143s 112ms/step - accuracy: 0.9528 - loss: 0.2126
Epoch 2/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 148s 117ms/step - accuracy: 0.9926 - loss: 0.0268
Epoch 3/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 136s 113ms/step - accuracy: 0.9945 - loss: 0.0170
Epoch 4/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 151s 121ms/step - accuracy: 0.9953 - loss: 0.0137
Epoch 5/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 151s 126ms/step - accuracy: 0.9959 - loss: 0.0118


In [10]:
# Step 9: Evaluate

# Evaluate the model on test data to check performance
loss, acc = model.evaluate(X_test, y_test)

# Print accuracy of the model
print("Accuracy:", acc)

300/300 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9933 - loss: 0.0203
Accuracy: 0.9933416247367859


In [12]:
# Step 10: Prediction

# Take a sample sentence from dataset
sample = sentences[0]

# Convert sentence to sequence and apply padding
seq = pad_sequences(tok.texts_to_sequences([sample]), maxlen=MAX_LEN, padding='post')

# Predict tag probabilities and get index of highest probability for each word
pred = np.argmax(model.predict(seq), axis=-1)[0]

# Create reverse mapping (index → tag)
idx2tag = {v:k for k,v in tag_tok.word_index.items()}

# Loop through each word and its predicted tag
for w, t in zip(sample, pred[:len(sample)]):
    # Print word with its predicted POS tag
    print(w, "->", idx2tag.get(t))

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Thousands -> nns
of -> in
demonstrators -> nns
have -> vbp
marched -> vbn
through -> in
London -> nnp
to -> to
protest -> vb
the -> dt
war -> nn
in -> in
Iraq -> nnp
and -> cc
demand -> nn
the -> dt
withdrawal -> nn
of -> in
British -> jj
troops -> nns
from -> in
that -> dt
country -> nn
. -> .


In [ ]:
1. What is LSTM?

LSTM (Long Short-Term Memory) is a type of RNN that can learn long-term dependencies in sequences.

2. Why use LSTM for POS tagging?

Because POS tagging depends on context, and LSTM can capture sequence relationships between words.

3. What is the input and output in this model?

Input = sequence of words
Output = sequence of POS tags

4. What is an Embedding layer?

It converts words into dense vector representations for better learning.

5. Why is padding required?

To make all sequences equal length for batch processing.

6. What is return_sequences=True in LSTM?

It ensures output is generated for every time step (needed for tagging each word).

7. What is TimeDistributed layer?

It applies the same Dense layer to each time step of the sequence.

8. Why use softmax activation?

To get probability distribution over all possible POS tags.

9. What loss function is used and why?

sparse_categorical_crossentropy because labels are integer encoded.

10. Why split data into train and test sets?

To evaluate model performance on unseen data.

🔸 Deep / Conceptual QnA
11. How does LSTM capture context?

By maintaining memory (cell state) that stores past information.

12. What problem does LSTM solve compared to simple RNN?

Vanishing gradient problem and inability to learn long-term dependencies.

13. What are the gates in LSTM?

Input gate, Forget gate, Output gate — control information flow.

14. Why is context important in POS tagging?

Because the same word can have different tags based on context.

15. What is overfitting in this model?

When model performs well on training data but poorly on test data.

16. How can you improve model accuracy?

Use more data, tuning hyperparameters, or using pretrained embeddings.

17. What are limitations of LSTM?

Slow training, high computation, struggles with very long sequences.

18. How is LSTM better than HMM (Viterbi)?

LSTM learns patterns automatically, while HMM depends on predefined probabilities.

19. Why do we use embeddings instead of one-hot encoding?

Embeddings capture semantic similarity and reduce dimensionality.

20. What is sequence labeling?

Assigning a label (tag) to each element in a sequence (word → POS tag).

🔥 High-Impact Viva Questions
21. Why not use simple RNN instead of LSTM?

Because RNN cannot handle long-term dependencies effectively.

22. What would happen if you remove padding?

Model cannot process variable-length sequences properly.

23. How would you handle unknown words?

Use OOV (out-of-vocabulary) tokens.

24. Can this model work for other tasks?

Yes, like NER, speech recognition, and machine translation.

25. How would you improve this model for real-world use?

Use pretrained models like BiLSTM, BERT, or add attention mechanisms.